# LandslideRisk-48 — reference solution
Plan: (1) load data, (2) build leak-free features (only current and *past* rows of each
location), (3) validate with a time-forward + unseen-location split that mimics the test
set, (4) compare a baseline against improved models, (5) refit the best model and predict.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

warnings.filterwarnings("ignore")
SEED = 48
TARGET = "landslide_risk_48h"
DATA = Path("./dataset/public")
OUT = Path("./working")
OUT.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(DATA / "train.csv", parse_dates=["timestamp"])
test = pd.read_csv(DATA / "test.csv", parse_dates=["timestamp"])
print(train.shape, test.shape)
print("train period:", train.timestamp.min(), "->", train.timestamp.max())
print("test period: ", test.timestamp.min(), "->", test.timestamp.max())
print("positive rate:", round(train[TARGET].mean(), 4))
unseen_test_locs = sorted(set(test.location_id) - set(train.location_id))
print("locations only in test:", unseen_test_locs)

## Metric
Same as the grader: mean of the log loss on seen-location rows and on unseen-location rows.

In [ ]:
def logloss(y, p):
    p = np.clip(p, 1e-15, 1 - 1e-15)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))

def balanced_logloss(y, p, unseen):
    return 0.5 * logloss(y[~unseen], p[~unseen]) + 0.5 * logloss(y[unseen], p[unseen])

## Features
Train and test are concatenated *only* to compute per-location lag features from earlier
rows (sensor history is known at prediction time). No labels are used, and nothing is
taken from future rows. `location_id` is deliberately not a model feature: it carries no
information for the locations that only appear in the test set.

In [ ]:
RAIN = ["rainfall_1h_mm", "rainfall_6h_mm", "rainfall_24h_mm", "rainfall_72h_mm"]

def build_features(df):
    df = df.sort_values(["location_id", "timestamp"]).copy()
    g = df.groupby("location_id")

    # Fill small sensor gaps with the location's previous reading (past only).
    for c in RAIN + ["soil_moisture", "vegetation_index", "temperature_c"]:
        df[c + "_ffill"] = g[c].ffill()

    doy = df.timestamp.dt.dayofyear
    df["doy_sin"], df["doy_cos"] = np.sin(2 * np.pi * doy / 365.25), np.cos(2 * np.pi * doy / 365.25)
    df["aspect_sin"], df["aspect_cos"] = np.sin(np.radians(df.aspect_deg)), np.cos(np.radians(df.aspect_deg))

    r1, r6, r24, r72 = (df[c + "_ffill"] for c in RAIN)
    df["rain_1_6h"], df["rain_6_24h"], df["rain_24_72h"] = r6 - r1, r24 - r6, r72 - r24
    df["rain_24_over_72"] = r24 / (r72 + 1.0)
    df["slope_x_moist"] = df.slope_deg * df.soil_moisture_ffill
    df["slope_x_rain72"] = df.slope_deg * r72
    df["moist_x_rain72"] = df.soil_moisture_ffill * r72
    df["sin_slope"] = np.sin(np.radians(df.slope_deg))

    # Antecedent conditions from earlier days at the same location.
    g = df.groupby("location_id")
    for c in ["rainfall_72h_mm_ffill", "soil_moisture_ffill"]:
        df[c + "_lag1"] = g[c].shift(1)
        df[c + "_lag3"] = g[c].shift(3)
        df[c + "_d1"] = df[c] - df[c + "_lag1"]
    df["rain24_sum7d"] = g["rainfall_24h_mm_ffill"].transform(lambda s: s.rolling(7, min_periods=1).sum())
    df["rain24_sum30d"] = g["rainfall_24h_mm_ffill"].transform(lambda s: s.rolling(30, min_periods=1).sum())
    df["moist_mean7d"] = g["soil_moisture_ffill"].transform(lambda s: s.rolling(7, min_periods=1).mean())

    df["soil_type"] = df.soil_type.astype(pd.CategoricalDtype(["sandy", "loam", "clay", "silt", "rocky", "peat"]))
    return df

full = build_features(pd.concat([train.assign(_split="train"), test.assign(_split="test")], ignore_index=True))
tr = full[full._split == "train"].reset_index(drop=True)
te = full[full._split == "test"].reset_index(drop=True)

BASE = ["elevation_m", "slope_deg", "aspect_deg", "soil_moisture", "soil_type", "vegetation_index",
        *RAIN, "temperature_c"]
ENGINEERED = BASE + ["doy_sin", "doy_cos", "aspect_sin", "aspect_cos", "rain_1_6h", "rain_6_24h",
        "rain_24_72h", "rain_24_over_72", "slope_x_moist", "slope_x_rain72", "moist_x_rain72", "sin_slope",
        "rainfall_72h_mm_ffill_lag1", "rainfall_72h_mm_ffill_lag3", "rainfall_72h_mm_ffill_d1",
        "soil_moisture_ffill_lag1", "soil_moisture_ffill_lag3", "soil_moisture_ffill_d1",
        "rain24_sum7d", "rain24_sum30d", "moist_mean7d"]

## Validation that mimics the test set
Random K-fold would leak future information and reward memorising locations. Instead each
fold trains on rows before 2025-05-01 (minus 6 held-out locations) and validates on
2025-05-01 -> 2025-08-31 for all locations. Three different location hold-outs reduce noise.

In [ ]:
VAL_START = pd.Timestamp("2025-05-01")
locs = np.sort(tr.location_id.unique())
folds = []
for k in range(3):
    held = np.random.default_rng(SEED + k).choice(locs, 6, replace=False)
    fit_idx = tr.index[(tr.timestamp < VAL_START) & ~tr.location_id.isin(held)]
    val_idx = tr.index[tr.timestamp >= VAL_START]
    folds.append((fit_idx, val_idx, tr.loc[val_idx, "location_id"].isin(held).to_numpy()))

LGB_PARAMS = dict(objective="binary", learning_rate=0.02, n_estimators=2000, num_leaves=4,
                  min_child_samples=200, subsample=0.8, subsample_freq=1, colsample_bytree=0.7,
                  reg_lambda=5.0, random_state=SEED, verbose=-1)

def make_lr(features):
    num = [c for c in features if c != "soil_type"]
    prep = ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
        ("soil", OneHotEncoder(handle_unknown="ignore"), ["soil_type"]),
    ])
    return make_pipeline(prep, LogisticRegression(max_iter=3000))

def evaluate_model(fit_predict):
    """Run fit_predict(fit_idx, val_idx) on every fold; return mean score and fold predictions."""
    scores, preds = [], []
    for fit_idx, val_idx, unseen in folds:
        p = fit_predict(fit_idx, val_idx)
        preds.append(p)
        scores.append(balanced_logloss(tr.loc[val_idx, TARGET].to_numpy(), p, unseen))
    return float(np.mean(scores)), preds

def lr_fp(features):
    def fp(fit_idx, val_idx):
        m = make_lr(features).fit(tr.loc[fit_idx, features], tr.loc[fit_idx, TARGET])
        return m.predict_proba(tr.loc[val_idx, features])[:, 1]
    return fp

best_iters = {}
def lgb_fp(features, name):
    def fp(fit_idx, val_idx):
        m = lgb.LGBMClassifier(**LGB_PARAMS)
        m.fit(tr.loc[fit_idx, features], tr.loc[fit_idx, TARGET],
              eval_set=[(tr.loc[val_idx, features], tr.loc[val_idx, TARGET])],
              callbacks=[lgb.early_stopping(100, verbose=False)])
        best_iters.setdefault(name, []).append(m.best_iteration_)
        return m.predict_proba(tr.loc[val_idx, features])[:, 1]
    return fp

### Step 1: baselines
A constant prior is the floor. Logistic regression and LightGBM on the raw columns are the
first real models.

In [ ]:
results, fold_preds = {}, {}
results["constant prior"], _ = evaluate_model(
    lambda f, v: np.full(len(v), tr.loc[f, TARGET].mean()))
results["logistic regression, raw"], _ = evaluate_model(lr_fp(BASE))
results["LightGBM, raw"], _ = evaluate_model(lgb_fp(BASE, "raw"))

### Step 2: does memorising locations help?
Adding `location_id` gives no gain on held-out locations, so it is left out.

In [ ]:
results["LightGBM, raw + location_id"], _ = evaluate_model(lgb_fp(BASE + ["location_id"], "loc"))

### Step 3: engineered features and a blend
Rainfall increments, antecedent lags, seasonality and slope x moisture interactions are added
to both models. The two model families make different errors, so a 50/50 blend is evaluated too.

In [ ]:
results["logistic regression, engineered"], fold_preds["lr"] = evaluate_model(lr_fp(ENGINEERED))
results["LightGBM, engineered"], fold_preds["lgb"] = evaluate_model(lgb_fp(ENGINEERED, "eng"))
results["blend 50/50, engineered"] = float(np.mean([
    balanced_logloss(tr.loc[v, TARGET].to_numpy(), 0.5 * a + 0.5 * b, u)
    for (f, v, u), a, b in zip(folds, fold_preds["lr"], fold_preds["lgb"])]))

# Overfitting check on the last fold: train vs validation log loss.
f, v, u = folds[-1]
n_eng = int(np.mean(best_iters["eng"]))
m = lgb.LGBMClassifier(**{**LGB_PARAMS, "n_estimators": n_eng}).fit(tr.loc[f, ENGINEERED], tr.loc[f, TARGET])
print("LightGBM train logloss:", round(logloss(tr.loc[f, TARGET].to_numpy(), m.predict_proba(tr.loc[f, ENGINEERED])[:, 1]), 4),
      "| validation logloss:", round(logloss(tr.loc[v, TARGET].to_numpy(), m.predict_proba(tr.loc[v, ENGINEERED])[:, 1]), 4))

summary = pd.Series(results, name="balanced log loss (lower is better)").round(4)
print(summary.to_string())

## Final model
Refit both models on all training rows (LightGBM uses the averaged early-stopping round count,
scaled for the larger training set, averaged over 5 seeds), blend 50/50, and clip predictions
away from 0 and 1 because one confident miss dominates log loss.

In [ ]:
lr_final = make_lr(ENGINEERED).fit(tr[ENGINEERED], tr[TARGET])
p_lr = lr_final.predict_proba(te[ENGINEERED])[:, 1]

p_lgb = np.zeros(len(te))
for s in range(5):
    m = lgb.LGBMClassifier(**{**LGB_PARAMS, "n_estimators": int(n_eng * 1.15), "random_state": SEED + s})
    m.fit(tr[ENGINEERED], tr[TARGET])
    p_lgb += m.predict_proba(te[ENGINEERED])[:, 1] / 5

preds = np.clip(0.5 * p_lr + 0.5 * p_lgb, 0.002, 0.998)

submission = pd.DataFrame({"record_id": te.record_id, TARGET: preds})
submission = submission.set_index("record_id").loc[test.record_id].reset_index()
assert len(submission) == len(test)
assert submission.record_id.is_unique and submission[TARGET].between(0, 1).all()
submission.to_csv(OUT / "submission.csv", index=False)
print(submission[TARGET].describe())